In [0]:
import zipfile
import json
from pyspark.sql.types import StructType
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

In [0]:
schema_path = "abfss://config@stgbbb.dfs.core.windows.net/schema"

In [0]:

def unzip_files(file):
    try:
        zip_name = file.name
        local_zip_path = f"/dbfs/tmp/{zip_name}"
        local_extract_path = f"/dbfs/tmp/extract_{zip_name.replace('.zip', '')}"
        
        dbutils.fs.cp(file.path, "file:" + local_zip_path)
        
        with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
            zip_ref.extractall(local_extract_path)
            
    except Exception as e:
        print(e)
    
    return local_extract_path
 

In [0]:

def schema_existe(nome_tabela, camada):
    try:
        dbutils.fs.ls(f"{schema_path}/{camada}/{nome_tabela}.json")
        return True
    except AnalysisException:
        return False

In [0]:
def save_schema(df, nome_tabela, camada):
    schema_json = df.schema.json()
    caminho = f"{schema_path}/{camada}/{nome_tabela}.json"
    dbutils.fs.put(caminho, schema_json, overwrite=True)

In [0]:
def enforce_schema(df, nome_tabela, camada):

    if not schema_existe(nome_tabela):
        save_schema(df, nome_tabela)
        return df

    """Lê o JSON do Lake e força o DF a seguir os tipos definidos."""
    caminho = f"{schema_path}/{camada}/{nome_tabela}.json"
    
    # 1. Ler o arquivo JSON do Storage
    json_str = dbutils.fs.head(caminho)
    stored_schema = StructType.fromJson(json.loads(json_str))
    
    stored_cols_map = {f.name: f.dataType for f in stored_schema}
    
    select_expr = []
    has_schema_changed = False
    
    for col_name in df.columns:
        
        if col_name in stored_cols_map:
            # A coluna JÁ EXISTIA: Forçamos o tipo antigo (Enforcement) para segurança
            target_type = stored_cols_map[col_name]
            # Ex: Se era Date e veio String, tenta converter. Se falhar, vira Null (Safe Cast)
            select_expr.append(F.col(col_name).cast(target_type).alias(col_name))
        else:
            # A coluna é NOVA: Deixamos passar como veio (Evolution)
            select_expr.append(F.col(col_name))
            has_schema_changed = True
    
    df_final = df.select(select_expr)
    
    if has_schema_changed:
        save_schema(df_final, nome_tabela)
        
    return df_final

In [0]:
### Função para tirar nulos e substituir por Nao Informado

def clean_and_fill(df, columns, replacement="NÃO INFORMADO"):
    from pyspark.sql.functions import when, col, trim
    """
    Transforma strings vazias em nulos e preenche nulos com um valor padrão.
    
    Args:
        df: DataFrame do Spark.
        columns: Lista de colunas para tratar (ex: ["regiao", "status"]).
        replacement: Texto que substituirá o nulo.
    """
    for column in columns:
        # Primeiro: trata espaços e transforma "" em None (nulo)
        df = df.withColumn(
            column, 
            when(trim(col(column)) == "", None).otherwise(col(column))
        )
    
    # Segundo: preenche todos os nulos das colunas selecionadas com o texto desejado
    df = df.fillna(replacement, subset=columns)   

    return df

In [0]:
### Função para transformar dados da colunas em minusculo e headers em maisculo
def lower_and_upper(df, columns):
    import pyspark.sql.functions as f
    #Transforma os DADOS das colunas em minúsculo
    for column in columns:
        df = df.withColumn(column, f.lower(f.col(column)))
    
    #Transforma os nomes das colunas em maiúsculo
    df = df.toDF(*[c.upper() for c in df.columns])
    
    return df

In [0]:

## Função para chamar todas as funções de tratamento de dados
def tratamento(df, colunas):
    df = clean_and_fill(df, colunas)
    df = lower_and_upper(df, colunas)
    return df 




In [0]:
### 3. Configuração de autenticação no Azure Data Lake
meu_storage = "stgbbb"
spark.conf.set(
    f"fs.azure.account.key.{meu_storage}.dfs.core.windows.net",
    "xzhcoxUXRj+GjVy8wOc3wladWad/Dm+OgQ5Lpj6U0RKd8K0+n18AJz12D3FmA/LTGP8SsbqSRybt+AStxQY00w=="
)

df = df_exportacao_estado_bronze = spark.read.parquet(
    "abfss://bronze@stgbbb.dfs.core.windows.net/cnpj/ano=2026/mes=02/dia=03/Estabelecimentos3/"
)

# df_final = tratamento(df, ['NO_NBM'])

df.count()